# Profit-Aware Threshold Evaluation

## Aim of this notebook

The purpose of this notebook is to evaluate whether the churn prediction models can support a financially useful customer-retention campaign.

The earlier model comparison showed how well the Random Forest and Azure AutoML models predicted churn. However, statistical measures such as accuracy, ROC-AUC and F1 score do not directly show whether using the model would produce business value. A model may perform well statistically but still create unnecessary costs if too many non-churners are contacted.

This notebook therefore evaluates a range of probability thresholds and measures the operational and financial consequences of each decision rule.

The analysis compares:

- the selected manual Random Forest model;
- the selected Azure AutoML model;
- the conventional threshold of 0.50;
- the profit-maximising threshold under three business scenarios.

The main outputs include:

- customers contacted;
- true churners contacted;
- false interventions;
- missed churners;
- expected retained-customer value;
- campaign cost;
- net profit;
- return on investment;
- profit per contacted customer;
- the recommended threshold for each scenario.

## Input file

The notebook uses the completed same-test prediction evidence file:

`E:\manual_automl_comparison_outputs\12_final_test_prediction_evidence.csv`

The required columns are:

- `actual_class`
- `manual_probability`
- `automl_probability`

The original prediction columns may also be present, but they are not required for the profit calculations.

## Methodological position

The preferred approach is to choose the business threshold using validation data and then apply that fixed threshold to an untouched test set.

In the present analysis, the available file contains the final test predictions from both models. Therefore, the threshold search is treated as a **post-hoc business sensitivity analysis** rather than an independently validated production policy.

This distinction is important. The results show how different thresholds would have changed the financial outcome on the observed test data, but the selected threshold should still be validated on future or newly collected data before operational deployment.

## 1. Import libraries and configure the analysis

The code below sets the exact location of the final prediction-evidence file.

The analysis tests thresholds from 0.00 to 1.00 in steps of 0.01. Three business scenarios are used to reflect uncertainty in customer value, campaign effectiveness and intervention cost.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RANDOM_STATE = 42
THRESHOLD_STEP = 0.01
DEFAULT_THRESHOLD = 0.50
N_BOOTSTRAP = 300

# Currency symbol is used only for labels.
CURRENCY_SYMBOL = "£"

# Set a full Path manually if automatic discovery fails.
TEST_PREDICTION_FILE = Path(
    r"E:\manual_automl_comparison_outputs\12_final_test_prediction_evidence.csv"
)

# Preferred: provide validation predictions here.
# Leave as None when only the test prediction file exists.
VALIDATION_PREDICTION_FILE = None

OUTPUT_DIR = (
    Path.cwd()
    / "profit_aware_evaluation_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output folder:", OUTPUT_DIR.resolve())

## 2. Locate and load the prediction evidence

The code validates the target and probability columns and confirms that probabilities lie between zero and one.

In [ ]:
def find_prediction_file(
    explicit_path=None,
    required_name=(
        "12_final_test_prediction_evidence.csv"
    ),
):
    if explicit_path is not None:
        explicit_path = Path(
            explicit_path
        )

        if not explicit_path.exists():
            raise FileNotFoundError(
                f"Prediction file not found: "
                f"{explicit_path}"
            )

        return explicit_path

    search_roots = [
        Path.cwd(),
        Path.home() / "Downloads",
        Path.home() / "Documents",
    ]

    matches = []

    for root in search_roots:
        if root.exists():
            matches.extend(
                root.rglob(
                    required_name
                )
            )

    unique_matches = sorted(
        {
            path.resolve()
            for path in matches
        }
    )

    if not unique_matches:
        raise FileNotFoundError(
            f"Could not find {required_name}. "
            "Set TEST_PREDICTION_FILE manually "
            "in Section 1."
        )

    if len(unique_matches) > 1:
        print(
            "Multiple prediction files found. "
            "Using the first one:"
        )

        for path in unique_matches:
            print(path)

    return unique_matches[0]


def load_prediction_evidence(
    file_path,
    dataset_name,
):
    frame = pd.read_csv(
        file_path
    )

    required_columns = {
        "actual_class",
        "manual_probability",
        "automl_probability",
    }

    missing_columns = (
        required_columns
        - set(frame.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{dataset_name} is missing columns: "
            + ", ".join(
                sorted(
                    missing_columns
                )
            )
        )

    frame = frame.copy()

    frame["actual_class"] = pd.to_numeric(
        frame["actual_class"],
        errors="raise",
    ).astype(int)

    probability_columns = [
        "manual_probability",
        "automl_probability",
    ]

    for column in probability_columns:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="raise",
        )

        if frame[column].isna().any():
            raise ValueError(
                f"{column} contains missing values."
            )

        if not frame[column].between(
            0,
            1,
            inclusive="both",
        ).all():
            raise ValueError(
                f"{column} contains values "
                "outside the range [0, 1]."
            )

    observed_classes = set(
        frame[
            "actual_class"
        ].unique()
    )

    if not observed_classes.issubset(
        {0, 1}
    ):
        raise ValueError(
            "actual_class must contain only 0 and 1."
        )

    print(
        f"{dataset_name}: "
        f"{len(frame):,} rows, "
        f"churn rate="
        f"{frame['actual_class'].mean():.4f}"
    )

    return frame


test_file = find_prediction_file(
    explicit_path=TEST_PREDICTION_FILE
)

test_predictions = load_prediction_evidence(
    file_path=test_file,
    dataset_name="Test prediction evidence",
)

if VALIDATION_PREDICTION_FILE is not None:
    validation_file = Path(
        VALIDATION_PREDICTION_FILE
    )

    validation_predictions = (
        load_prediction_evidence(
            file_path=validation_file,
            dataset_name=(
                "Validation prediction evidence"
            ),
        )
    )

    selection_predictions = (
        validation_predictions
    )

    analysis_mode = (
        "validation_selected_test_evaluated"
    )

else:
    validation_file = None

    selection_predictions = (
        test_predictions
    )

    analysis_mode = (
        "exploratory_same_test_sensitivity"
    )

print("\nTest file:", test_file)
print("Analysis mode:", analysis_mode)

display(
    test_predictions.head()
)

## 3. Business assumptions

The financial assumptions used in this notebook are transparent scenario values rather than verified company figures.

Each scenario includes:

- `retained_customer_value`: the estimated contribution value gained when a churner is successfully retained;
- `intervention_success_rate`: the probability that the retention action prevents churn;
- `cost_per_contact`: the cost of contacting one selected customer and providing the offer;
- `false_positive_penalty`: an additional allowance for unnecessarily targeting a non-churner;
- `fixed_campaign_cost`: any fixed campaign cost.

Three scenarios are used:

- **Conservative:** lower retained value and lower campaign effectiveness;
- **Expected:** central assumptions;
- **Optimistic:** higher retained value and higher campaign effectiveness.

These assumptions should be replaced with organisation-specific or literature-supported values when stronger evidence becomes available.

In [ ]:
BUSINESS_SCENARIOS = {
    "conservative": {
        "retained_customer_value": 60.0,
        "intervention_success_rate": 0.10,
        "cost_per_contact": 8.0,
        "false_positive_penalty": 1.0,
        "fixed_campaign_cost": 0.0,
    },
    "expected": {
        "retained_customer_value": 120.0,
        "intervention_success_rate": 0.20,
        "cost_per_contact": 8.0,
        "false_positive_penalty": 0.50,
        "fixed_campaign_cost": 0.0,
    },
    "optimistic": {
        "retained_customer_value": 180.0,
        "intervention_success_rate": 0.30,
        "cost_per_contact": 8.0,
        "false_positive_penalty": 0.0,
        "fixed_campaign_cost": 0.0,
    },
}

assumption_rows = []

for scenario_name, assumptions in (
    BUSINESS_SCENARIOS.items()
):
    assumption_rows.append(
        {
            "scenario": scenario_name,
            **assumptions,
            "expected_value_per_true_positive": (
                assumptions[
                    "retained_customer_value"
                ]
                * assumptions[
                    "intervention_success_rate"
                ]
            ),
        }
    )

assumptions_table = pd.DataFrame(
    assumption_rows
)

assumptions_table.to_csv(
    OUTPUT_DIR
    / "01_profit_assumptions.csv",
    index=False,
)

display(
    assumptions_table
)

## 4. Profit calculation

For each threshold:

- customers with a predicted churn probability greater than or equal to the threshold are contacted;
- true positives are actual churners who are contacted;
- false positives are non-churners who are contacted;
- false negatives are churners who are not contacted.

The expected retained value is calculated as:

**True positives × intervention success rate × retained customer value**

The total campaign cost is calculated as:

**contact cost + false-positive penalty + fixed campaign cost**

The net profit is then calculated as:

**expected retained value − total campaign cost**

This structure makes the financial assumptions visible and allows the effect of alternative scenarios to be tested.

In [ ]:
MODEL_COLUMNS = {
    "Random Forest": "manual_probability",
    "Azure AutoML selected model": (
        "automl_probability"
    ),
}


def safe_divide(
    numerator,
    denominator,
):
    if denominator == 0:
        return np.nan

    return (
        numerator
        / denominator
    )


def calculate_profit_row(
    actual,
    probability,
    threshold,
    model_name,
    scenario_name,
    assumptions,
    evaluation_dataset,
):
    prediction = (
        probability
        >= threshold
    ).astype(int)

    actual = np.asarray(
        actual,
        dtype=int,
    )

    true_positive = int(
        np.sum(
            (actual == 1)
            & (prediction == 1)
        )
    )

    false_positive = int(
        np.sum(
            (actual == 0)
            & (prediction == 1)
        )
    )

    false_negative = int(
        np.sum(
            (actual == 1)
            & (prediction == 0)
        )
    )

    true_negative = int(
        np.sum(
            (actual == 0)
            & (prediction == 0)
        )
    )

    contacted = (
        true_positive
        + false_positive
    )

    expected_retained_customers = (
        true_positive
        * assumptions[
            "intervention_success_rate"
        ]
    )

    expected_retained_value = (
        expected_retained_customers
        * assumptions[
            "retained_customer_value"
        ]
    )

    contact_cost = (
        contacted
        * assumptions[
            "cost_per_contact"
        ]
    )

    false_positive_cost = (
        false_positive
        * assumptions[
            "false_positive_penalty"
        ]
    )

    fixed_campaign_cost = (
        assumptions[
            "fixed_campaign_cost"
        ]
        if contacted > 0
        else 0.0
    )

    total_campaign_cost = (
        contact_cost
        + false_positive_cost
        + fixed_campaign_cost
    )

    net_profit = (
        expected_retained_value
        - total_campaign_cost
    )

    roi = safe_divide(
        net_profit,
        total_campaign_cost,
    )

    sample_size = len(
        actual
    )

    total_churners = int(
        np.sum(
            actual == 1
        )
    )

    return {
        "evaluation_dataset": (
            evaluation_dataset
        ),
        "model": model_name,
        "scenario": scenario_name,
        "threshold": float(
            threshold
        ),
        "sample_size": int(
            sample_size
        ),
        "total_churners": (
            total_churners
        ),
        "customers_contacted": int(
            contacted
        ),
        "contact_rate": (
            safe_divide(
                contacted,
                sample_size,
            )
        ),
        "true_churners_contacted": (
            true_positive
        ),
        "false_interventions": (
            false_positive
        ),
        "missed_churners": (
            false_negative
        ),
        "true_nonchurners_not_contacted": (
            true_negative
        ),
        "campaign_precision": (
            safe_divide(
                true_positive,
                contacted,
            )
        ),
        "churn_capture_rate": (
            safe_divide(
                true_positive,
                total_churners,
            )
        ),
        "expected_retained_customers": (
            expected_retained_customers
        ),
        "expected_retained_value": (
            expected_retained_value
        ),
        "contact_cost": (
            contact_cost
        ),
        "false_positive_cost": (
            false_positive_cost
        ),
        "fixed_campaign_cost": (
            fixed_campaign_cost
        ),
        "total_campaign_cost": (
            total_campaign_cost
        ),
        "net_profit": (
            net_profit
        ),
        "roi": roi,
        "profit_per_contacted_customer": (
            safe_divide(
                net_profit,
                contacted,
            )
        ),
        "profit_per_1000_customers": (
            net_profit
            / sample_size
            * 1000
        ),
    }

## 5. Evaluate all thresholds

Thresholds from 0.00 to 1.00 are tested at the configured interval.

A threshold of 0.00 contacts every customer.  
A threshold of 1.00 generally represents a no-contact or near-no-contact policy.

In [ ]:
thresholds = np.round(
    np.arange(
        0.00,
        1.00
        + THRESHOLD_STEP,
        THRESHOLD_STEP,
    ),
    10,
)

profit_rows = []

selection_actual = (
    selection_predictions[
        "actual_class"
    ]
    .to_numpy()
)

for model_name, probability_column in (
    MODEL_COLUMNS.items()
):
    probability = (
        selection_predictions[
            probability_column
        ]
        .to_numpy()
    )

    for scenario_name, assumptions in (
        BUSINESS_SCENARIOS.items()
    ):
        for threshold in thresholds:
            profit_rows.append(
                calculate_profit_row(
                    actual=selection_actual,
                    probability=probability,
                    threshold=threshold,
                    model_name=model_name,
                    scenario_name=scenario_name,
                    assumptions=assumptions,
                    evaluation_dataset=(
                        "validation"
                        if validation_file
                        is not None
                        else "test_exploratory"
                    ),
                )
            )

profit_by_threshold = pd.DataFrame(
    profit_rows
)

profit_by_threshold.to_csv(
    OUTPUT_DIR
    / "02_profit_by_threshold.csv",
    index=False,
)

print(
    "Threshold evaluations:",
    f"{len(profit_by_threshold):,}",
)

display(
    profit_by_threshold.head()
)

## 6. Select the profit-maximising threshold

For each model and scenario, the selected threshold maximises net profit.

When multiple thresholds produce the same net profit, the tie is resolved by:

1. contacting fewer customers;
2. using the higher threshold.

This favours the less costly operational policy when profit is equal.

In [ ]:
optimal_threshold_rows = []

for (
    model_name,
    scenario_name,
), group in profit_by_threshold.groupby(
    [
        "model",
        "scenario",
    ]
):
    selected_row = (
        group.sort_values(
            by=[
                "net_profit",
                "customers_contacted",
                "threshold",
            ],
            ascending=[
                False,
                True,
                False,
            ],
        )
        .iloc[0]
        .copy()
    )

    optimal_threshold_rows.append(
        selected_row
    )

optimal_thresholds = pd.DataFrame(
    optimal_threshold_rows
).reset_index(
    drop=True
)

optimal_thresholds.to_csv(
    OUTPUT_DIR
    / "03_optimal_thresholds_by_scenario.csv",
    index=False,
)

display_columns = [
    "model",
    "scenario",
    "threshold",
    "customers_contacted",
    "contact_rate",
    "true_churners_contacted",
    "false_interventions",
    "missed_churners",
    "campaign_precision",
    "churn_capture_rate",
    "expected_retained_value",
    "total_campaign_cost",
    "net_profit",
    "roi",
]

display(
    optimal_thresholds[
        display_columns
    ].style.format(
        {
            "threshold": "{:.2f}",
            "contact_rate": "{:.2%}",
            "campaign_precision": "{:.2%}",
            "churn_capture_rate": "{:.2%}",
            "expected_retained_value": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "total_campaign_cost": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "roi": "{:.2f}",
        }
    )
)

## 7. Evaluate selected thresholds on the final test data

When a separate validation prediction file is supplied, this section applies the validation-selected thresholds to the untouched test set.

In exploratory mode, it reproduces the selected test sensitivity results and clearly labels them as post-hoc.

In [ ]:
test_evaluation_rows = []

test_actual = (
    test_predictions[
        "actual_class"
    ]
    .to_numpy()
)

for _, selected_row in (
    optimal_thresholds.iterrows()
):
    model_name = selected_row[
        "model"
    ]

    scenario_name = selected_row[
        "scenario"
    ]

    threshold = float(
        selected_row[
            "threshold"
        ]
    )

    probability_column = (
        MODEL_COLUMNS[
            model_name
        ]
    )

    assumptions = (
        BUSINESS_SCENARIOS[
            scenario_name
        ]
    )

    test_evaluation_rows.append(
        calculate_profit_row(
            actual=test_actual,
            probability=(
                test_predictions[
                    probability_column
                ]
                .to_numpy()
            ),
            threshold=threshold,
            model_name=model_name,
            scenario_name=scenario_name,
            assumptions=assumptions,
            evaluation_dataset="test",
        )
    )

selected_threshold_test_results = (
    pd.DataFrame(
        test_evaluation_rows
    )
)

selected_threshold_test_results[
    "threshold_selection_source"
] = (
    "validation"
    if validation_file is not None
    else "same_test_exploratory"
)

selected_threshold_test_results.to_csv(
    OUTPUT_DIR
    / "04_selected_threshold_test_results.csv",
    index=False,
)

display(
    selected_threshold_test_results[
        display_columns
        + [
            "threshold_selection_source"
        ]
    ].style.format(
        {
            "threshold": "{:.2f}",
            "contact_rate": "{:.2%}",
            "campaign_precision": "{:.2%}",
            "churn_capture_rate": "{:.2%}",
            "expected_retained_value": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "total_campaign_cost": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "roi": "{:.2f}",
        }
    )
)

## 8. Compare the default 0.50 threshold with the profit-aware threshold

This comparison shows whether the business-optimised threshold improves net profit relative to the conventional 0.50 classification threshold.

In [ ]:
comparison_rows = []

for _, selected_result in (
    selected_threshold_test_results
    .iterrows()
):
    model_name = selected_result[
        "model"
    ]

    scenario_name = selected_result[
        "scenario"
    ]

    probability_column = (
        MODEL_COLUMNS[
            model_name
        ]
    )

    assumptions = (
        BUSINESS_SCENARIOS[
            scenario_name
        ]
    )

    default_result = (
        calculate_profit_row(
            actual=test_actual,
            probability=(
                test_predictions[
                    probability_column
                ]
                .to_numpy()
            ),
            threshold=(
                DEFAULT_THRESHOLD
            ),
            model_name=model_name,
            scenario_name=(
                scenario_name
            ),
            assumptions=assumptions,
            evaluation_dataset=(
                "test_default_threshold"
            ),
        )
    )

    comparison_rows.append(
        {
            "model": model_name,
            "scenario": scenario_name,
            "default_threshold": (
                DEFAULT_THRESHOLD
            ),
            "selected_threshold": (
                selected_result[
                    "threshold"
                ]
            ),
            "default_net_profit": (
                default_result[
                    "net_profit"
                ]
            ),
            "selected_net_profit": (
                selected_result[
                    "net_profit"
                ]
            ),
            "profit_improvement": (
                selected_result[
                    "net_profit"
                ]
                - default_result[
                    "net_profit"
                ]
            ),
            "default_customers_contacted": (
                default_result[
                    "customers_contacted"
                ]
            ),
            "selected_customers_contacted": (
                selected_result[
                    "customers_contacted"
                ]
            ),
            "default_churn_capture_rate": (
                default_result[
                    "churn_capture_rate"
                ]
            ),
            "selected_churn_capture_rate": (
                selected_result[
                    "churn_capture_rate"
                ]
            ),
        }
    )

default_vs_selected = pd.DataFrame(
    comparison_rows
)

default_vs_selected.to_csv(
    OUTPUT_DIR
    / "05_default_vs_profit_aware_threshold.csv",
    index=False,
)

display(
    default_vs_selected.style.format(
        {
            "default_threshold": "{:.2f}",
            "selected_threshold": "{:.2f}",
            "default_net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "selected_net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "profit_improvement": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "default_churn_capture_rate": (
                "{:.2%}"
            ),
            "selected_churn_capture_rate": (
                "{:.2%}"
            ),
        }
    )
)

## 9. Identify robust near-optimal thresholds

A threshold is treated as near-optimal when its net profit is at least 95% of the maximum profit for that model and scenario.

This produces a practical range rather than presenting one threshold as perfectly precise.

In [ ]:
robust_threshold_rows = []

for (
    model_name,
    scenario_name,
), group in profit_by_threshold.groupby(
    [
        "model",
        "scenario",
    ]
):
    maximum_profit = group[
        "net_profit"
    ].max()

    if maximum_profit > 0:
        near_optimal = group.loc[
            group[
                "net_profit"
            ]
            >= 0.95
            * maximum_profit
        ]
    else:
        near_optimal = group.loc[
            group[
                "net_profit"
            ]
            >= maximum_profit
        ]

    robust_threshold_rows.append(
        {
            "model": model_name,
            "scenario": (
                scenario_name
            ),
            "maximum_net_profit": (
                maximum_profit
            ),
            "near_optimal_threshold_min": (
                near_optimal[
                    "threshold"
                ].min()
            ),
            "near_optimal_threshold_max": (
                near_optimal[
                    "threshold"
                ].max()
            ),
            "number_of_near_optimal_thresholds": (
                len(
                    near_optimal
                )
            ),
        }
    )

robust_thresholds = pd.DataFrame(
    robust_threshold_rows
)

robust_thresholds.to_csv(
    OUTPUT_DIR
    / "06_robust_threshold_ranges.csv",
    index=False,
)

display(
    robust_thresholds.style.format(
        {
            "maximum_net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "near_optimal_threshold_min": (
                "{:.2f}"
            ),
            "near_optimal_threshold_max": (
                "{:.2f}"
            ),
        }
    )
)

## 10. Plot profit curves

One figure is produced for each business scenario. The vertical dotted lines mark the selected thresholds.

In [ ]:
for scenario_name in (
    BUSINESS_SCENARIOS
):
    figure, axis = plt.subplots(
        figsize=(10, 6)
    )

    scenario_data = (
        profit_by_threshold.loc[
            profit_by_threshold[
                "scenario"
            ]
            == scenario_name
        ]
    )

    for model_name in (
        MODEL_COLUMNS
    ):
        model_data = (
            scenario_data.loc[
                scenario_data[
                    "model"
                ]
                == model_name
            ]
            .sort_values(
                "threshold"
            )
        )

        axis.plot(
            model_data[
                "threshold"
            ],
            model_data[
                "net_profit"
            ],
            label=model_name,
        )

        selected_threshold = float(
            optimal_thresholds.loc[
                (
                    optimal_thresholds[
                        "model"
                    ]
                    == model_name
                )
                & (
                    optimal_thresholds[
                        "scenario"
                    ]
                    == scenario_name
                ),
                "threshold",
            ].iloc[0]
        )

        axis.axvline(
            selected_threshold,
            linestyle=":",
            alpha=0.6,
        )

    axis.axhline(
        0,
        linewidth=1,
    )

    axis.set_title(
        "Net profit by probability threshold "
        f"({scenario_name.title()} scenario)"
    )

    axis.set_xlabel(
        "Probability threshold"
    )

    axis.set_ylabel(
        f"Net profit ({CURRENCY_SYMBOL})"
    )

    axis.legend()

    axis.grid(
        alpha=0.3
    )

    figure.tight_layout()

    figure_path = (
        OUTPUT_DIR
        / (
            "07_profit_curve_"
            f"{scenario_name}.png"
        )
    )

    figure.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Saved:",
        figure_path.resolve(),
    )

## 11. Bootstrap uncertainty at the selected thresholds

Bootstrap resampling estimates uncertainty around net profit after the threshold has been selected.

The intervals quantify sampling uncertainty in the evaluation data. They do not capture uncertainty in the business assumptions themselves.

In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_rows = []

sample_size = len(
    test_predictions
)

for _, selected_result in (
    selected_threshold_test_results
    .iterrows()
):
    model_name = selected_result[
        "model"
    ]

    scenario_name = selected_result[
        "scenario"
    ]

    threshold = float(
        selected_result[
            "threshold"
        ]
    )

    probability_column = (
        MODEL_COLUMNS[
            model_name
        ]
    )

    assumptions = (
        BUSINESS_SCENARIOS[
            scenario_name
        ]
    )

    actual_array = (
        test_predictions[
            "actual_class"
        ]
        .to_numpy()
    )

    probability_array = (
        test_predictions[
            probability_column
        ]
        .to_numpy()
    )

    bootstrap_profits = []

    for bootstrap_iteration in range(
        N_BOOTSTRAP
    ):
        sampled_indices = (
            rng.integers(
                0,
                sample_size,
                size=sample_size,
            )
        )

        bootstrap_result = (
            calculate_profit_row(
                actual=actual_array[
                    sampled_indices
                ],
                probability=(
                    probability_array[
                        sampled_indices
                    ]
                ),
                threshold=threshold,
                model_name=model_name,
                scenario_name=(
                    scenario_name
                ),
                assumptions=(
                    assumptions
                ),
                evaluation_dataset=(
                    "bootstrap_test"
                ),
            )
        )

        bootstrap_profits.append(
            bootstrap_result[
                "net_profit"
            ]
        )

    bootstrap_profits = np.asarray(
        bootstrap_profits
    )

    bootstrap_rows.append(
        {
            "model": model_name,
            "scenario": scenario_name,
            "threshold": threshold,
            "bootstrap_iterations": (
                N_BOOTSTRAP
            ),
            "net_profit_lower_95": (
                np.quantile(
                    bootstrap_profits,
                    0.025,
                )
            ),
            "net_profit_median": (
                np.median(
                    bootstrap_profits
                )
            ),
            "net_profit_upper_95": (
                np.quantile(
                    bootstrap_profits,
                    0.975,
                )
            ),
            "probability_profit_positive": (
                np.mean(
                    bootstrap_profits
                    > 0
                )
            ),
        }
    )

bootstrap_profit_intervals = (
    pd.DataFrame(
        bootstrap_rows
    )
)

bootstrap_profit_intervals.to_csv(
    OUTPUT_DIR
    / "08_bootstrap_profit_intervals.csv",
    index=False,
)

display(
    bootstrap_profit_intervals.style.format(
        {
            "threshold": "{:.2f}",
            "net_profit_lower_95": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "net_profit_median": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "net_profit_upper_95": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "probability_profit_positive": (
                "{:.2%}"
            ),
        }
    )
)

## 12. Produce the recommended-threshold summary

This table identifies the model with the highest test net profit under each scenario.

The recommendation remains conditional on the business assumptions. A lower-recall model may maximise profit when unnecessary interventions are expensive, while a higher-recall model may be preferred when missed churners are especially costly.

In [ ]:
recommendation_rows = []

for scenario_name, group in (
    selected_threshold_test_results
    .groupby(
        "scenario"
    )
):
    best_row = (
        group.sort_values(
            by=[
                "net_profit",
                "customers_contacted",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .iloc[0]
    )

    recommendation_rows.append(
        {
            "scenario": scenario_name,
            "recommended_model": (
                best_row[
                    "model"
                ]
            ),
            "recommended_threshold": (
                best_row[
                    "threshold"
                ]
            ),
            "expected_net_profit": (
                best_row[
                    "net_profit"
                ]
            ),
            "customers_contacted": (
                best_row[
                    "customers_contacted"
                ]
            ),
            "contact_rate": (
                best_row[
                    "contact_rate"
                ]
            ),
            "churn_capture_rate": (
                best_row[
                    "churn_capture_rate"
                ]
            ),
            "campaign_precision": (
                best_row[
                    "campaign_precision"
                ]
            ),
            "threshold_selection_source": (
                best_row[
                    "threshold_selection_source"
                ]
            ),
        }
    )

recommended_business_thresholds = (
    pd.DataFrame(
        recommendation_rows
    )
)

recommended_business_thresholds.to_csv(
    OUTPUT_DIR
    / "09_recommended_business_thresholds.csv",
    index=False,
)

display(
    recommended_business_thresholds.style.format(
        {
            "recommended_threshold": (
                "{:.2f}"
            ),
            "expected_net_profit": (
                CURRENCY_SYMBOL
                + "{:,.2f}"
            ),
            "contact_rate": "{:.2%}",
            "churn_capture_rate": (
                "{:.2%}"
            ),
            "campaign_precision": (
                "{:.2%}"
            ),
        }
    )
)

## 13. Produce a dissertation-ready conclusion

The final cell summarises the recommended model and threshold under each scenario.

The conclusion deliberately states that the results depend on the business assumptions and that the threshold search was conducted as a post-hoc sensitivity analysis using the available test predictions.

In [ ]:
conclusion_lines = [
    "PROFIT-AWARE THRESHOLD EVALUATION",
    "",
    f"Analysis mode: {analysis_mode}",
    (
        "Threshold step: "
        f"{THRESHOLD_STEP:.3f}"
    ),
    (
        "Default comparison threshold: "
        f"{DEFAULT_THRESHOLD:.2f}"
    ),
    "",
]

for _, row in (
    recommended_business_thresholds
    .sort_values(
        "scenario"
    )
    .iterrows()
):
    conclusion_lines.append(
        (
            f"{row['scenario'].title()} scenario: "
            f"{row['recommended_model']} at "
            f"threshold "
            f"{row['recommended_threshold']:.2f}; "
            f"expected net profit "
            f"{CURRENCY_SYMBOL}"
            f"{row['expected_net_profit']:,.2f}; "
            f"contact rate "
            f"{row['contact_rate']:.2%}; "
            f"churn capture rate "
            f"{row['churn_capture_rate']:.2%}."
        )
    )

conclusion_lines.extend(
    [
        "",
        (
            "The financial results are conditional "
            "on the retained-customer value, "
            "intervention effectiveness and campaign "
            "cost assumptions."
        ),
        (
            "They should therefore be interpreted as "
            "scenario-based decision support rather "
            "than audited company profit forecasts."
        ),
    ]
)

if analysis_mode == (
    "exploratory_same_test_sensitivity"
):
    conclusion_lines.extend(
        [
            "",
            (
                "Methodological limitation: the same "
                "test prediction file was used to "
                "identify and evaluate the "
                "profit-maximising thresholds."
            ),
            (
                "The thresholds are therefore post-hoc "
                "sensitivity results and should be "
                "validated on future or separate "
                "validation data before deployment."
            ),
        ]
    )

conclusion_text = "\n".join(
    conclusion_lines
)

print(
    conclusion_text
)

(
    OUTPUT_DIR
    / "10_profit_aware_conclusion.txt"
).write_text(
    conclusion_text,
    encoding="utf-8",
)

## 14. Output verification

The final folder should contain all tables, figures, assumptions and the written conclusion required for the dissertation evidence package.

In [ ]:
expected_outputs = [
    "01_profit_assumptions.csv",
    "02_profit_by_threshold.csv",
    "03_optimal_thresholds_by_scenario.csv",
    "04_selected_threshold_test_results.csv",
    "05_default_vs_profit_aware_threshold.csv",
    "06_robust_threshold_ranges.csv",
    "07_profit_curve_conservative.png",
    "07_profit_curve_expected.png",
    "07_profit_curve_optimistic.png",
    "08_bootstrap_profit_intervals.csv",
    "09_recommended_business_thresholds.csv",
    "10_profit_aware_conclusion.txt",
]

output_verification = pd.DataFrame(
    {
        "file": expected_outputs,
        "exists": [
            (
                OUTPUT_DIR
                / file_name
            ).exists()
            for file_name in (
                expected_outputs
            )
        ],
    }
)

display(
    output_verification
)

if output_verification[
    "exists"
].all():
    print(
        "\nPROFIT-AWARE EVALUATION "
        "COMPLETED SUCCESSFULLY"
    )

    print(
        "Output folder:",
        OUTPUT_DIR.resolve(),
    )

else:
    missing_outputs = (
        output_verification.loc[
            ~output_verification[
                "exists"
            ],
            "file",
        ]
        .tolist()
    )

    raise FileNotFoundError(
        "Missing expected outputs: "
        + ", ".join(
            missing_outputs
        )
    )